# Retrieval

LangChain에서 Retrieval은 외부 데이터에서 관련 정보를 찾아 프롬프트에 포함시켜(Context) LLM에 전달하는 역할을 한다. 주요 구성 요소는 다음과 같다.

- **Document Loader**: 다양한 원본 데이터를 LangChain 표준 문서 객체로 변환한다.
- **Text Splitter**: 긴 문서를 작은 청크로 분할해 검색 효율을 높인다.
- **Embedding Model**: 텍스트를 의미 기반 벡터로 변환한다.
- **Vector Store**: 임베딩된 벡터를 저장하고 유사도 기반 검색을 지원한다.
- **Retriever**: 쿼리에 대해 관련 문서를 찾아주는 표준 인터페이스를 제공한다.
- **Retrieval Chain**: 검색된 문서를 LLM에 전달해 답변을 생성하는 체인 구조를 제공한다.

이렇게 각 모듈이 결합되어, 외부 데이터 기반의 효과적인 검색 및 답변 생성이 가능하다.

**환각 Hallucination:**

LLM이 실제 근거 없이 그럴듯해 보이는 정보를 생성하는 현상이다.

**주요 원인**
1. **학습 데이터 한계**
   * 모델이 학습한 데이터에 해당 정보가 없거나 부족할 때 발생한다.
2. **확률적 생성 과정**
   * 토큰 예측 시 언어적 일관성을 우선하다 보니, 사실 여부가 검증되지 않은 내용을 생성한다.
3. **프롬프트 모호성**
   * 지시가 불명확하거나 맥락이 부족하면 모델이 관련 없는 정보를 보충·왜곡한다.

**대표 사례**
* 존재하지 않는 논문·저자명을 인용함.
* 역사적·과학적 사실을 잘못 기술함.
* 실행 불가능하거나 비효율적인 코드 제안.


**완화 방안**

1. **지식 기반 검색 결합**
   * Retrieval-Augmented Generation(RAG) 방식으로 외부 문서·데이터베이스에서 실시간 근거를 가져와 보강한다.
2. **프롬프트 구체화**
   * “출처를 함께 제시해 달라” 등 명시적 요청을 통해 근거 표기를 유도한다.
3. **후처리 검증**
   * 생성 결과를 룰 기반 검증 또는 전문가 리뷰를 통해 교차 확인한다.
4. **모델 파인튜닝 및 앙상블**
   * 도메인 특화 데이터로 추가 학습하거나, 룰 기반 시스템과 결합하여 정확도를 높인다.

In [2]:
!pip install langchain langchain-community langchain-openai langchain-huggingface wikipedia pypdf tavily-python tiktoken faiss-cpu sentence-transformers -Uqqq

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

## Document

Document는 LangChain 프레임워크에서 다양한 데이터 소스(예: 텍스트 파일, PDF, 웹페이지 등)로부터 불러온 정보를 표준화된 객체로 표현하는 핵심 데이터 구조이다. 이 객체는 언어 모델(LLM)이 외부 데이터를 이해하고 처리할 수 있도록 도와준다.

**Document 객체의 구조**
1. page_content: 문서의 실제 내용을 담고 있는 문자열(str)이다. 예를 들어, 텍스트 파일의 본문이나 PDF의 텍스트 등이 여기에 저장된다.
2. metadata: 문서에 대한 부가 정보를 담는 딕셔너리(dict) 형태의 속성이다. 예를 들어, 파일 경로, 페이지 번호, 작성자, 데이터 출처 등 다양한 메타데이터를 저장할 수 있다.


**Document의 역할과 활용**
1. 표준화된 데이터 구조: 다양한 포맷의 데이터를 일관된 방식으로 표현하여, LLM이 손쉽게 접근하고 활용할 수 있도록 한다.
2. 문서 처리의 기본 단위: LangChain의 문서 로더(Document Loader)는 파일, 웹, 데이터베이스 등 여러 소스에서 데이터를 읽어와 Document 객체로 변환한다.
3. 청크 단위 분할: 대용량 문서는 작은 단위(청크)로 쪼개어 각각의 Document로 저장하고, 검색 및 임베딩 처리에 활용한다.

In [ ]:
from langchain_core.documents import Document  # Langchain 표준 문서 단위 객체 

doc = Document(
    # 문서 본문 텍스트
    page_content='이것은 랭체인의 Document 객체입니다. 모든 데이터베이스는 이 Document 객체로 변환됩니다.',
    # 문서에 붙는 부가정보
    metadata = {
        'source' : '여기저기',  # 데이터 출처
        'url' : 'https://encore.com',  # 원문 url
        'timestamp' : 2020608250939  # 수집/생성 시간 
    }
)

print(doc)
print(doc.page_content)
print(doc.metadata)

page_content='이것은 랭체인의 Document 객체입니다. 모든 데이터베이스는 이 Document 객체로 변환됩니다.' metadata={'source': '여기저기', 'url': 'https://encore.com', 'timestamp': 2020608250939}
이것은 랭체인의 Document 객체입니다. 모든 데이터베이스는 이 Document 객체로 변환됩니다.
{'source': '여기저기', 'url': 'https://encore.com', 'timestamp': 2020608250939}


## Document Loader
https://reference.langchain.com/python/langchain_core/document_loaders/


Document Loader는 다양한 데이터 소스에서 데이터를 읽어와 Document 객체로 변환하는 역할을 한다. 예를 들어, PDFLoader, CSVLoader, TextLoader 등 다양한 종류가 존재하며, 각기 다른 파일 형식을 Document 객체로 표준화한다.

Document Loader는 데이터 소스별로 특화된 클래스를 제공하며, 문서를 로드한 후 LangChain에서 사용하는 표준 형식으로 변환해준다.

1. **다양한 데이터 소스 지원**  
   Document Loader는 파일 시스템, 클라우드 스토리지, 데이터베이스, 웹 등 다양한 데이터 소스에서 데이터를 로드할 수 있도록 설계되었다.
   
2. **표준화된 출력 형식**  
   로드된 문서는 LangChain에서 사용하는 `Document` 객체로 변환된다. `Document` 객체는 다음과 같은 필드를 포함한다:
   - `page_content`: 문서 본문 내용
   - `metadata`: 문서와 관련된 메타데이터 (예: 파일 이름, URL, 작성자 등)

3. **플러그인 기반 확장 가능**  
   사용자 정의 데이터 소스 로더를 쉽게 구현하고 LangChain에 통합할 수 있다.

**주요 Document Loader 예시**

| Loader 이름        | 설명                                                              |
|--------------------|-------------------------------------------------------------------|
| `PyPDFLoader`      | PDF 문서를 로드하며 텍스트를 추출해 Document 형식으로 변환한다.     |
| `TextLoader`       | 일반 텍스트 파일을 로드한다.                                      |
| `UnstructuredFileLoader` | 비구조적 데이터를 로드하여 구조화된 텍스트로 변환한다.           |
| `CSVLoader`        | CSV 파일에서 데이터를 로드하며 행(row)을 Document로 처리한다.      |
| `WebBaseLoader`    | 웹 페이지 데이터를 크롤링하여 Document로 로드한다.                |

In [ ]:
from langchain_community.document_loaders import WebBaseLoader  # url을 주면 document로 뽑아주는 

url = 'https://n.news.naver.com/article/015/0005324329' 

header = {
    # 브라우저 식별 : Windows에서 Chrome으로 접속한 것처럼 보이게 만드는 UA
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

loader = WebBaseLoader(url, header_template=header)  # 로더 객체 생성
docs = loader.load()  # 불러온 웹페이지 -> Document 리스트
docs

c:\Users\Playdata\LLM\llm_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://n.news.naver.com/article/015/0005324329', 'title': "문자도 전서구처럼 느리게…'슬로테크' 메시지 앱 인기", 'language': 'ko'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n문자도 전서구처럼 느리게…\'슬로테크\' 메시지 앱 인기\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n본문 바로가기\n\n\n\n\n\n\n이전 페이지\n\n\n\n\n\n\n\n\n\n\n한국경제\n\n\n\n\n\n구독\n\n메인 뉴스판에서 한국경제 주요뉴스를 볼 수 있습니다.\n보러가기\n닫기\n\n\n한국경제 언론사 구독 해지되었습니다.\n닫기\n\n\n\n\n\n\n\n\n\n\n주요뉴스\n이슈\n클립\n정치\n경제\n사회\nIT\n생활\n세계\n사설/칼럼\n신문보기\n생중계\n랭킹\n\n\n\nMY\n\n\n뉴스 이용 설정을 할 수 있어요\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n한국경제\n\n\n\n문자도 전서구처럼 느리게…\'슬로테크\' 메시지 앱 인기\n\n\n\n\n\n\n\n\n입력\n2026.08.25. 오전 9:46\n\n\n수정\n2026.08.25. 오전 9:49\n\n\n\n기사원문\n \n\n\n\n\n\n\n\n\n\n\n추천\n반응\n\n\n\n\n쏠쏠정보\n0\n\n\n\n\n흥미진진\n0\n\n\n\n\n공감백배\n0\n\n\n\n\n분석탁월\n0\n\n\n\n\n후속강추\n0\n\n\n \n\n\n\n\n댓글\n반응\n\n\n\n\n\n\n\n\n텍스트 음성 변환 서비스 사용하기\n\n\n\n성별\n남성\n여성\n\n\n말하기 속도\n느림\n보통\n빠름\n\n이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다.\n본문듣기 시작\n\n닫기\n\n\n \n\n글자 크기 변경하기\n\n글자크기\n\n\n가1단계\n작게

In [ ]:
print(len(docs))

doc = docs[0]
print(doc.metadata)  # 메타데이터
print(doc.metadata['title'])  # 메타데이터 title
print(doc.page_content.replace('\n', ''))  # 본문 내용

1
{'source': 'https://n.news.naver.com/article/015/0005324329', 'title': "문자도 전서구처럼 느리게…'슬로테크' 메시지 앱 인기", 'language': 'ko'}
문자도 전서구처럼 느리게…'슬로테크' 메시지 앱 인기
문자도 전서구처럼 느리게…'슬로테크' 메시지 앱 인기본문 바로가기이전 페이지한국경제구독메인 뉴스판에서 한국경제 주요뉴스를 볼 수 있습니다.보러가기닫기한국경제 언론사 구독 해지되었습니다.닫기주요뉴스이슈클립정치경제사회IT생활세계사설/칼럼신문보기생중계랭킹MY뉴스 이용 설정을 할 수 있어요한국경제문자도 전서구처럼 느리게…'슬로테크' 메시지 앱 인기입력2026.08.25. 오전 9:46수정2026.08.25. 오전 9:49기사원문 추천반응쏠쏠정보0흥미진진0공감백배0분석탁월0후속강추0 댓글반응텍스트 음성 변환 서비스 사용하기성별남성여성말하기 속도느림보통빠름이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다.본문듣기 시작닫기 글자 크기 변경하기글자크기가1단계작게가2단계보통가3단계크게가4단계아주크게가5단계최대크게닫기SNS 보내기인쇄하기캐리어피지·루스트 이용자 급증비둘기·달팽이 등 동물 속도로 전송실종·사망 확률 0.2%…현실감 더해사진=게티이미지뱅크19세기 주요 통신수단이었던 ‘귀소비둘기’(전서구)가 최근 스마트폰을 통해 부활했다. 메시지를 극도로 느린 속도로 전송하는 애플리케이션(앱)이 인기를 끌면서다. 전서구뿐 아니라 펭귄, 달팽이 등 동물의 이동속도를 흉내 낸 것으로, 디지털 피로에 지친 이용자의 마음을 사로잡았다는 평가가 나온다.뉴욕타임스(NYT)에 따르면 지난 4월 출시된 저속 메시징 앱인 ‘캐리어 피지’(편지 나르는 비둘기)는 지난 18일 기준 이용자 7만6000명을 기록했다. 개발자인 노아 이야로비노가 지난 12일 앱을 소개하는 게시글을 올린 이후 이용자가 급증했다. 전송 속도는 전서구 종류에 따라 다르며, 최대 시속 110마일(약 177㎞) 수준이다. 지도에서

In [10]:
from pprint import pprint

pprint(doc.page_content.replace('\n', ''))

("문자도 전서구처럼 느리게…'슬로테크' 메시지 앱 인기본문 바로가기이전 페이지한국경제구독메인 뉴스판에서 한국경제 주요뉴스를 볼 수 "
 '있습니다.보러가기닫기한국경제 언론사 구독 해지되었습니다.닫기주요뉴스이슈클립정치경제사회IT생활세계사설/칼럼신문보기생중계랭킹MY뉴스 이용 '
 "설정을 할 수 있어요한국경제문자도 전서구처럼 느리게…'슬로테크' 메시지 앱 인기입력2026.08.25. 오전 "
 '9:46수정2026.08.25. 오전 9:49기사원문 추천반응쏠쏠정보0흥미진진0공감백배0분석탁월0후속강추0 댓글반응텍스트 음성 변환 '
 '서비스 사용하기성별남성여성말하기 속도느림보통빠름이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다.본문듣기 '
 '시작닫기 글자 크기 변경하기글자크기가1단계작게가2단계보통가3단계크게가4단계아주크게가5단계최대크게닫기SNS 보내기인쇄하기캐리어피지·루스트 '
 '이용자 급증비둘기·달팽이 등 동물 속도로 전송실종·사망 확률 0.2%…현실감 더해사진=게티이미지뱅크19세기 주요 통신수단이었던 '
 '‘귀소비둘기’(전서구)가 최근 스마트폰을 통해 부활했다. 메시지를 극도로 느린 속도로 전송하는 애플리케이션(앱)이 인기를 끌면서다. '
 '전서구뿐 아니라 펭귄, 달팽이 등 동물의 이동속도를 흉내 낸 것으로, 디지털 피로에 지친 이용자의 마음을 사로잡았다는 평가가 '
 '나온다.뉴욕타임스(NYT)에 따르면 지난 4월 출시된 저속 메시징 앱인 ‘캐리어 피지’(편지 나르는 비둘기)는 지난 18일 기준 이용자 '
 '7만6000명을 기록했다. 개발자인 노아 이야로비노가 지난 12일 앱을 소개하는 게시글을 올린 이후 이용자가 급증했다. 전송 속도는 '
 '전서구 종류에 따라 다르며, 최대 시속 110마일(약 177㎞) 수준이다. 지도에서 새가 목적지를 향해 날아가는 모습을 추적할 수 '
 '있다.비둘기가 길을 잃거나 죽을 가능성을 0.2%로 설정해 현실감을 더했다. 앱 이용자인 올리비아 맥도널드(27)는 “비둘기가 사라질 '


In [13]:
# 톰소여의 여행(영문)
!pip install gdown
!gdown 1o7ngiyeJJ-MPLhl0fiCKHViTNNpk6zjO

Downloading...
From: https://drive.google.com/uc?id=1o7ngiyeJJ-MPLhl0fiCKHViTNNpk6zjO
To: c:\Users\Playdata\LLM\05_langchain\02_langchain_component\The_Adventures_of_Tom_Sawyer.pdf

  0%|          | 0.00/2.68M [00:00<?, ?B/s]
 20%|█▉        | 524k/2.68M [00:00<00:00, 2.42MB/s]
 59%|█████▊    | 1.57M/2.68M [00:00<00:00, 5.44MB/s]
100%|██████████| 2.68M/2.68M [00:00<00:00, 7.13MB/s]


In [14]:
from langchain_community.document_loaders import PyPDFLoader  # PDF를 읽어 Document로 로드하는 로더

loader = PyPDFLoader('The_Adventures_of_Tom_Sawyer.pdf')
docs = loader.load()  # PDF를 페이지별 Document 리스트로 변환
print(len(docs))  # Document 수

35


In [15]:
print(docs[2].metadata)  # 3페이지의 메타데이터
print(docs[2].metadata['source'])  # 경로
print(docs[2].metadata['page'])  # 현재 페이지
print(docs[2].metadata['page_label'])  # 사람이 실제로 볼 수 있는 페이지 번호
print(docs[2].page_content)  # 본문

{'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 2, 'page_label': '3'}
The_Adventures_of_Tom_Sawyer.pdf
2
3
The Adventures of                 
Tom Sawyer 
 
MARK TWAIN 
Level 1 
 
Retold by Jacqueline Kehl                                                    
Series Editors: Andy Hopkins and Jocelyn Potter


In [16]:
print(docs[5].page_content)  # 6페이지 내용

Chapter 1    The Fence 
 
Tom Sawyer lived with his aunt because his mother and 
father were dead. Tom didn’t like going to school, and he 
didn’t like working. He liked playing and having 
adventures. One Friday, he didn’t go to school—he went 
to the river. 
Aunt Polly was angry. “You’re a bad boy!” she said. 
“Tomorrow you can’t play with your friends because you 
didn’t go to school today. Tomorrow you’re going to work 
for me. You can paint the fence.” 
Saturday morning, Tom was not happy, but he started to 
paint the fence. His friend Jim was in the street. 
Tom asked him, “Do you want to paint?” 
Jim said, “No, I can’t. I’m going to get water.” 
Then Ben came to Tom’s house. He watched Tom and 
said, “I’m going to swim today. You can’t swim because 
you’re working.” 
Tom said, “This isn’t work. I like painting.” 
“Can I paint, too?” Ben asked. 
“No, you can’t,” Tom answered. “Aunt Polly asked me 
because I’m a very good painter.” 
Ben said, “I’m a good painter, too. Please, can 

### TavilySearchAPIRetriever
https://www.tavily.com/

- `langchain_tavily.TavilySearch`: Agent tool사용버젼. json반환
- `langchain_community.retrievers.TavilySearchAPIRetriever`: 검색기(context확보용) Document객체반환

- 주요 기능
    - 웹 검색(query → 결과 리스트): 키워드로 웹을 검색해서 관련 페이지들을 찾아줌
    - 요약/스니펫 제공: 각 결과에 본문 요약이나 핵심 스니펫을 같이 줘서 LLM이 바로 쓰기 좋음
    - 컨텐츠 추출(include_raw_content 등 옵션): 결과 페이지의 내용을 일부/전체 텍스트로 가져오게 설정 가능
    - 필터링/튜닝 옵션: 검색 결과 개수, 도메인 포함/제외, 최신성(리센시) 같은 옵션으로 결과를 조절 가능
    - RAG 파이프라인에 바로 연결: “검색 → 문서(Document)화 → 벡터화/리랭킹 → 답변” 흐름에서 검색 단계로 많이 사용

In [17]:
%pip install tavily-python -qqq

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from dotenv import load_dotenv  # .env 환경변수 로드
import os

load_dotenv()  # .env 파일 읽어와 환경변수로 등록
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")  # .env의 값으로 TAVILY_API_KEY 등록

In [ ]:
# Tavily 웹검색 결과를 받아 Document로 반환하는 Retriever
from langchain_community.retrievers import TavilySearchAPIRetriever

tavily_retriever = TavilySearchAPIRetriever(k=3)  # 검색결과 상위 3개

docs = tavily_retriever.invoke('휴가')  # 검색어로 웹 검색 -> Document 리스트
docs

[Document(metadata={'title': '휴가', 'source': 'https://namu.wiki/w/%ED%9C%B4%EA%B0%80', 'score': 1.0, 'id': '210fd3-00', 'images': []}, page_content='최근 변경최근 토론\n\n특수 기능\n\n\n\n\n\n여야 합니다. 해당 문서의 ACL 탭을 확인하시기 바랍니다.\n\n 편집 권한이 부족한 경우 아래의 \'편집 요청\' 버튼으로 편집 요청을 생성할 수 있습니다. (\n\n편집 요청닫기 \n\n토론역사\n\n분류\n   휴가\n   조직관리\n   복지\n\nImage 2Image 3: 다른 뜻 아이콘 다른 뜻에 대한 내용은 휴가(동음이의어) "휴가(동음이의어)") 문서를 참고하십시오.\n\n1. 개요2. 학생3. 직장인\n\n3.1. 유럽 및 남미3.2. 미국3.3. 대한민국\n\n3.3.1. 비정규직의 휴가\n\n4. 휴가종류5. 군인\n\n5.1. 대한민국 국군5.2. 군인의 휴가 준비5.3. 종류 및 그에 따른 특징과 규정\n\n5.3.1. 정기 휴가5.3.2. 특별 휴가5.3.3. 청원 휴가\n\n5.4. 대체복무5.5. 북한군의 휴가\n\n6. 종류\n\n## 1.개요( [...] 6. 종류\n\n## 1.개요(\n\n휴가(休暇, vacation(미)/holiday(영)/leave)은 직장 "직장(장소)")·학교·군대 따위의 단체에서, 일정한 기간 동안 쉬는 일 또는 그런 겨를이다. (표준국어대사전 정의)\n\n## 2.학생( [...] ## 2.학생(\n\n학생은 방학이라고 하여 1년에 2번 총 60일 정도의 정기 휴가 제도가 있다. 대학생쯤 되면 방학은 아니지만 학기 중간중간 아무것도 없는 빈 기간이 생기기도 한다. 주로 종강 이후 다시 개강하기까지의 빈 기간을 방학이라고 하는 편이다. 대학생들은 방학 때 마냥 놀기보다 취업 준비나 아르바이트 등으로 인해서 오히려 굉장히 바쁘게 지내는 경우가 많다. 고3도 여름방학은 보통 대입 준비

In [20]:
for doc in docs:
    print(doc.page_content)

최근 변경최근 토론

특수 기능





여야 합니다. 해당 문서의 ACL 탭을 확인하시기 바랍니다.

 편집 권한이 부족한 경우 아래의 '편집 요청' 버튼으로 편집 요청을 생성할 수 있습니다. (

편집 요청닫기 

토론역사

분류
   휴가
   조직관리
   복지

Image 2Image 3: 다른 뜻 아이콘 다른 뜻에 대한 내용은 휴가(동음이의어) "휴가(동음이의어)") 문서를 참고하십시오.

1. 개요2. 학생3. 직장인

3.1. 유럽 및 남미3.2. 미국3.3. 대한민국

3.3.1. 비정규직의 휴가

4. 휴가종류5. 군인

5.1. 대한민국 국군5.2. 군인의 휴가 준비5.3. 종류 및 그에 따른 특징과 규정

5.3.1. 정기 휴가5.3.2. 특별 휴가5.3.3. 청원 휴가

5.4. 대체복무5.5. 북한군의 휴가

6. 종류

## 1.개요( [...] 6. 종류

## 1.개요(

휴가(休暇, vacation(미)/holiday(영)/leave)은 직장 "직장(장소)")·학교·군대 따위의 단체에서, 일정한 기간 동안 쉬는 일 또는 그런 겨를이다. (표준국어대사전 정의)

## 2.학생( [...] ## 2.학생(

학생은 방학이라고 하여 1년에 2번 총 60일 정도의 정기 휴가 제도가 있다. 대학생쯤 되면 방학은 아니지만 학기 중간중간 아무것도 없는 빈 기간이 생기기도 한다. 주로 종강 이후 다시 개강하기까지의 빈 기간을 방학이라고 하는 편이다. 대학생들은 방학 때 마냥 놀기보다 취업 준비나 아르바이트 등으로 인해서 오히려 굉장히 바쁘게 지내는 경우가 많다. 고3도 여름방학은 보통 대입 준비에 박차를 가하는 기간이 된다.( "대학 진학을 희망하지 않는 경우에는 고3 여름방학도 노는 기간이 된다. 다만 특성화고등학교, 마이스터고등학교는 취업 준비 때문에 여름방학에 놀지 못할 수 있다.") 고2 겨울방학은 예비고3이라고 하며 이때부터 대입 준비를 본격적으로 하는 경우도 많다.

## 3.직장인(
전담 지원센터 : 1670-1330
 휴가샵 온

### Tavily 검색 결과 로 Context에 넣고 답변하는 RAG Chain

In [26]:
from langchain_core.prompts import PromptTemplate  # 프롬프트 템플릿 생성
from langchain.chat_models import init_chat_model  # 랭체인 ChatModel 구성
from langchain_core.output_parsers import StrOutputParser  # LLM 응답 -> 문자열
from langchain_core.runnables import RunnablePassthrough  # 입력을 그대로 통과시키는 Runnable
from langchain_core.documents import Document  # Document 객체

tavily_retriever = TavilySearchAPIRetriever(k=3)  # 검색결과 상위 3개
prompt = PromptTemplate.from_template('''
    사용자의 질문에 Context 기반으로 답변하세요. 모르는 내용은 모른다고 답변하세요.
    Context : {context}
    Question : {question}
''')  # 프롬프트 지정

llm = init_chat_model('openai:gpt-5.6-luna')  # 사용할 LLM 모델 지정
output_parser = StrOutputParser()  # 최종 출력은 텍스트 형식

# 검색된 Document 리스트를 프롬프트에 넣기 좋은 문자열 형태로 합치는 함수
def format_docs(docs: list[Document]) -> str:
    return '\n\n'.join(doc.page_content for doc in docs)  # 문서 본문끼리 개행문자로 이어붙인 문자열

tavily_chain = tavily_retriever | format_docs  # (질문) -> 검색 -> 텍스트

# question은 입력값 그대로 전달, context는 tavily 검색 결과 | 최종 프롬프트 완성 | LLM 호출 | 문자열 파싱
chain = (
    {'question' : RunnablePassthrough(), 'context' : tavily_chain} | prompt | llm |output_parser
)

print(chain.invoke('8월 말 독산역 인기 맛집?'))

8월 말 독산역 근처에서 방문하기 좋은 맛집은 다음과 같습니다. 제공된 정보 기준으로 추천드리며, 실제 영업 여부와 대기 상황은 방문 전 확인하는 것이 좋습니다.

### 독산역 1번 출구 인근

- **우마왕**  
  소고기 특수부위와 삼합 전문. 독산역 1번 출구에서 약 9분 거리로, 고기 메뉴를 찾을 때 좋습니다.  
  ☎ 0507-1303-3911

- **실비순대국**  
  순대국과 순대스테이크를 함께 즐길 수 있는 곳. 약 10분 거리입니다.  
  ☎ 02-804-8856

- **삼은식당**  
  토시살·치마살·제비추리 등 소고기 특수부위 전문. 약 13분 거리입니다.  
  ☎ 02-807-1587

- **협진식당 5호점**  
  등심과 갈비살 전문이며 주차가 가능합니다. 약 13분 거리입니다.  
  ☎ 02-805-3403

- **장수삼계탕**  
  약삼계탕 전문점으로, 8월 말 보양식이 생각날 때 적합합니다. 약 13분 거리이며 **일요일 휴무**입니다.  
  ☎ 02-2169-2096

- **서대문김치찜**  
  삼겹살과 김치찜을 함께 즐길 수 있는 곳. 약 14분 거리입니다.  
  ☎ 02-2104-6656

- **지리산삼계탕**  
  들깨삼계탕과 전복삼계탕 메뉴가 있으며 약 15분 거리입니다. **수요일 휴무**입니다.  
  ☎ 02-869-2009

- **이화룡불짬뽕**  
  돼지·차돌·해물짬뽕을 취급하는 중식당으로, 매운 국물 음식이 당길 때 추천합니다. 약 19분 거리입니다.  
  ☎ 02-864-0025

### 독산역 2번 출구·가산 방향

- **제주세호해장국**  
  독산역 2번 출구에서 약 3분 거리로 접근성이 가장 좋습니다. 해장국 식사를 원할 때 추천합니다.  
  ☎ 02-6138-2322

- **도원**  
  중식 뷔페로, 여러 메뉴를 한 번에 먹고 싶을 때 적합합니다. 2번 출구에서 약 4분 거리입니다.  
  ☎ 0507-1337-6616

### 목적별 추천

- **8

## Embedding Model
- openai
- setence-transformer(huggingface)

In [ ]:
from langchain_openai import OpenAIEmbeddings
import pandas as pd

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')  # 임베딩 모델 (1536 차원)

text = '철수는 골든리트리버를 키우고 있습니다.'

emb_vec = embeddings.embed_query(text)  # 텍스트 임베딩 -> float 리스트(벡터) 변환
print(len(emb_vec))  # 임베딩 차원 수
print(emb_vec[:3])  # 샘플 3개 확인

pd.Series(emb_vec, name='embedding')

1536
[-0.0273590087890625, -0.00527191162109375, 0.005352020263671875]


0      -0.027359
1      -0.005272
2       0.005352
3      -0.049652
4       0.026611
          ...   
1531    0.022980
1532   -0.026749
1533   -0.001171
1534    0.036957
1535   -0.023468
Name: embedding, Length: 1536, dtype: float64

In [29]:
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd

embeddings = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')  # 임베딩 모델 (1536 차원)

text = '철수는 골든리트리버를 키우고 있습니다.'

emb_vec = embeddings.embed_query(text)  # 텍스트 임베딩 -> float 리스트(벡터) 변환
print(len(emb_vec))  # 임베딩 차원 수
print(emb_vec[:3])  # 샘플 3개 확인

pd.Series(emb_vec, name='embedding')

c:\Users\Playdata\LLM\llm_venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13489.88it/s]


384
[0.011021507903933525, 0.06504975259304047, 0.048188380897045135]


0      0.011022
1      0.065050
2      0.048188
3     -0.068704
4      0.014492
         ...   
379    0.067011
380    0.011610
381    0.025551
382   -0.018868
383    0.005621
Name: embedding, Length: 384, dtype: float64

## Vector Store

벡터 데이터베이스란 쉽게 말해, **비정형 데이터(텍스트, 이미지, 오디오 등)를 숫자 벡터로 변환하여 저장하고, 이 벡터들 간의 유사성을 바탕으로 데이터를 검색**하는 데이터베이스를 말한다. 여기서 벡터는 데이터를 다차원 공간에서 표현한 수학적 객체이다.

- **벡터**: 데이터의 특징을 다차원으로 표현한 값.
  - 예: 단어 임베딩은 단어를 벡터로 변환하여 유사한 단어들이 가까이 위치.
- **벡터 데이터베이스 필요성**:
  - RDBMS는 구조화된 데이터(테이블 형태)에 적합.
  - AI/머신러닝의 발전으로 비정형 데이터를 처리할 필요 증가.
  - 벡터 데이터베이스는 **유사도 기반 검색**으로 고차원 데이터 처리에 유리.

**주요 특징:**
- 유사한 데이터를 빠르게 검색.
- AI 응용 분야(이미지 검색, 자연어 처리, 추천 시스템 등)에서 중요.

- **벡터 데이터베이스와 RDBMS의 주요 차이점**

| **특징**                | **RDBMS**                                                                 | **벡터 데이터베이스**                                                                                  |
|-------------------------|---------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------|
| **데이터 구조**          | 테이블 형식으로 데이터 저장, SQL을 사용하여 질의.                             | 다차원 벡터 형식으로 데이터 저장, 벡터 간 유사도 계산 방식 사용.                                        |
| **검색 방식**            | 키-값 쌍이나 고정 조건 기반 검색 (정확한 일치 검색).                          | 유사성 검색 수행, 벡터 간 거리(예: 코사인 유사도, 유클리드 거리)를 기준으로 유사한 데이터를 반환.         |
| **비정형 데이터 처리**   | 텍스트, 숫자 등 구조화된 데이터 처리에 적합.                                 | 이미지, 오디오, 영상 등 비정형 데이터를 벡터로 변환해 처리 가능.                                       |
| **응용 분야**            | 전통적인 CRUD 작업, 금융 데이터, 고객 데이터 관리 등.                       | AI 기반 추천 시스템, 이미지 검색, 자연어 처리, 음성 인식 등.                                           |
| **확장성**               | 수평 확장 가능하지만 고차원 데이터나 복잡한 쿼리 처리에는 한계.               | 수백만~수십억 개 벡터 데이터를 효율적으로 처리 가능.                                                  |

**벡터 데이터베이스의 주요 특징**

1. **Approximate Nearest Neighbor (ANN) 검색**  
   - **ANN 알고리즘**을 사용해 유사한 벡터를 빠르게 검색.  
   - 검색 속도가 빠르고, 대규모 데이터셋에서도 효율적으로 동작.

2. **확장성**  
   - 수백만~수십억 개의 벡터 데이터를 처리할 수 있는 구조로 설계.  
   - 대규모 데이터셋에서 고속 검색 및 처리가 가능.

3. **유연성**  
   - 텍스트, 이미지, 오디오 데이터를 임베딩 형태로 변환해 저장 가능.  
   - 다양한 머신러닝 모델과 통합하여 사용자 요구에 맞는 검색 시스템 구축 가능.

**주요 벡터 데이터베이스 비교**

| **이름**      | **특징**                                                                                                                                   | **장점**                                                                                                             | **단점**                                                                                  |
|---------------|-------------------------------------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------|
| **Chroma**    | 오픈 소스 벡터 데이터베이스, LLM(대규모 언어 모델) 응용에 최적화. Python 노트북 환경에서 간편하게 사용 가능하며 프로덕션으로 확장 가능.                | 간편한 설정, 유사성 검색 및 임베딩 관리 용이, LLM 응용 프로그램에 적합.                                               | 대규모 데이터 처리에서 다른 서비스만큼 최적화되어 있지 않을 수 있음.                                          |
| **Pinecone**  | 완전 관리형 서비스로, 대규모 고차원 데이터의 실시간 처리 및 검색에 최적화.                                                                  | 유지보수 불필요(관리형 서비스), 실시간 대규모 데이터 검색에 강점, 데이터 엔지니어 및 과학자들에게 적합.                 | 오픈 소스가 아니며, 서비스 사용 비용이 발생.                                                              |
| **Weaviate**  | 오픈 소스 기반, OpenAI, Cohere, HuggingFace와의 통합으로 벡터화 작업 용이.                                                                 | 다양한 플랫폼과의 통합 기능, 확장성과 유연성, 고차원 데이터 검색 성능 우수.                                           | 복잡한 설정 및 사용 시 초기 학습 필요.                                                              |
| **Faiss**     | Meta에서 개발한 라이브러리로 대규모 벡터 세트 검색에 최적화. Python 및 GPU 지원으로 성능 극대화.                                              | 고성능 검색(GPU 지원), 대규모 데이터셋 처리 능력, 빠른 속도.                                                          | 데이터베이스가 아닌 라이브러리 형태로 제공되어, 추가적인 환경 설정 및 통합 작업 필요.                                         |
| **Qdrant**    | Rust로 구현된 API 기반 벡터 검색 도구. 빠른 검색과 자원 최적화를 제공하며 정교한 필터링 기능 지원.                                             | 뛰어난 성능(Rust 기반), 정교한 필터링 기능, API 중심의 유연한 설계.                                                    | 커뮤니티와 생태계가 다른 데이터베이스에 비해 상대적으로 작을 수 있음.                                         |

**선택 가이드**
1. **LLM 응용 프로그램**: Chroma, Weaviate.  
2. **완전 관리형 서비스**: Pinecone.  
3. **고성능 및 GPU 지원 필요**: Faiss.  
4. **정교한 필터링과 최적화된 성능**: Qdrant.  

### FAISS

- **공식 문서**: https://faiss.ai/
- **Github**: https://github.com/facebookresearch/faiss

**Faiss(Vector Search Library)**는 Facebook AI Research에서 개발한 **효율적인 벡터 검색 및 밀집 벡터 인덱싱 라이브러리**이다. 대규모 데이터에서 **빠른 유사도 검색과 군집화**를 수행하는 데 최적화되어 있다. 주로 문서 검색, 추천 시스템, 이미지 검색, NLP 모델에서 벡터 임베딩 처리를 지원한다.

**주요 특징**
1. **효율적인 유사도 검색**
   - `k-NN (k-Nearest Neighbors)`를 기반으로 벡터 간 유사도(예: 코사인 유사도, L2 거리)를 계산한다.
   - CPU/GPU 모두 지원하여 대규모 데이터에서도 빠르게 처리 가능하다.

2. **고성능 인덱싱**
   - 다양한 **인덱싱 알고리즘**(Flat, IVF, HNSW, PQ 등)을 지원하여 정확도와 속도 간 균형을 맞출 수 있다.
   - 데이터가 커질수록 효율적으로 검색 성능을 발휘하도록 설계되었다.

3. **확장성**
   - 수억 개의 벡터에서도 성능을 유지하도록 설계되었으며, GPU 병렬 처리를 통해 성능을 극대화한다.

4. **유연성**
   - Python과 C++ API를 제공하며, Scikit-learn이나 PyTorch와 같은 다른 라이브러리와 통합하여 사용 가능하다.

**Faiss의 기본 인덱스 유형**
1. **Flat Index**
   - 모든 벡터를 저장하고 전체 탐색(Brute-Force)을 수행.
   - 정확도가 높지만 대규모 데이터에서는 속도가 느릴 수 있다.

2. **IVF (Inverted File Index)**
   - 벡터를 클러스터링하여 데이터 양을 줄이고 탐색 속도를 높임.
   - 대규모 데이터에서 적합하며, 정확도와 속도 조절 가능.

3. **PQ (Product Quantization)**
   - 벡터를 압축하여 메모리 사용량을 줄이고, 빠른 근사 유사도 검색 수행.

4. **HNSW (Hierarchical Navigable Small World Graphs)**
   - 그래프 기반 알고리즘으로 매우 빠른 근사 유사도 검색 가능.


**Faiss의 주요 사용 사례**
1. **문서 검색**
   - 문서를 벡터로 변환한 후 가장 관련 있는 문서를 검색.
   - NLP 모델의 임베딩과 결합하여 사용.

2. **이미지 검색**
   - 이미지 특징 벡터를 사용하여 비슷한 이미지를 검색.

3. **추천 시스템**
   - 사용자의 행동이나 관심사를 벡터화하여 추천 품목 생성.

4. **클러스터링**
   - 벡터 데이터를 군집화하여 데이터의 구조를 분석.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
import numpy as np

loader = PyPDFLoader('The_Adventures_of_Tom_Sawyer.pdf')  # PDF 로더 객체 생성
docs = loader.load()  # PDF -> 페이지 단위 Document
page_contents = [doc.page_content for doc in docs]  # 각 페이지의 텍스트를 리스트로 추출

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')  # 1536 차원
emb_vecs = embeddings.embed_documents(page_contents)  # 페이지별 컨텐츠 임베딩 -> 벡터 리스트
np.array(emb_vecs).shape  # (페이지 수, 임베딩 차원 수)

(35, 1536)

In [ ]:
# FAISS 벡터스토어를 이용해 문서들을 임베딩해 로컬에 저장
from langchain_community.vectorstores import FAISS  # FAISS 기반 벡터 DB (= Vector Store)

vector_db = FAISS.from_documents(docs, embeddings)  # docs를 임베딩해서 FAISS의 인덱스 생성
vector_db.save_local('./db/faiss')  # 로컬 경로에 FAISS 인덱스/메타데이터 저장

In [ ]:
# 로컬에 저장해놓은 FAISS 벡터스토어 로드
vector_db = FAISS.load_local(
    './db/faiss',  # 경로
    embeddings,  # 로드 시 사용할 임베딩 모델
    allow_dangerous_deserialization=True  # 신뢰된 파일만 사용 (pickle 역직렬화 허용)
)

In [ ]:
search_result = vector_db.similarity_search(
    query='학교 선생님이 아끼는 해부학 책은 누가 찢었는가?',  # 쿼리: 한글
    k=4  # 상위 4개 Document
)

search_result

[Document(id='ee1caf77-45d1-4b30-bb61-d5d609176887', metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 15, 'page_label': '16'}, page_content='talking about it. Becky wanted to talk to Tom, but he \ndidn’t look at her. \nThen Tom talked to Amy. Becky watched him and she \nwas angry. She said to her friends, “I’m going to have an \nadventure day. You can come on my adventure.” But she \ndidn’t ask Tom. \nLater in the morning, Tom ta lked to Amy again. Becky \ntalked to her friend Alfred and looked at a picture-book \nwith him. Tom watched them and he was angry with \nBecky. \nIn the afternoon, Tom waited for Becky at the school \nfence. He said, “I’m sorry.” \nBut Beck

In [ ]:
for i, doc in enumerate(search_result, 1):
    # 사람이 보는 페이지 : 본문 내용
    print(f"{i}번째 {doc.metadata['page_label']} \npage : {doc.page_content}")

1번째 16 
page : talking about it. Becky wanted to talk to Tom, but he 
didn’t look at her. 
Then Tom talked to Amy. Becky watched him and she 
was angry. She said to her friends, “I’m going to have an 
adventure day. You can come on my adventure.” But she 
didn’t ask Tom. 
Later in the morning, Tom ta lked to Amy again. Becky 
talked to her friend Alfred and looked at a picture-book 
with him. Tom watched them and he was angry with 
Becky. 
In the afternoon, Tom waited for Becky at the school 
fence. He said, “I’m sorry.” 
But Becky didn’t listen to him. She walked into the 
school room. The teacher’s new book was on his table. 
This book wasn’t for children, but Becky wanted to look 
at it. She opened the book quietly and looked at the 
pictures. 
Suddenly, Tom came into the room. Becky was 
surprised. She closed the book quickly, and it tore. Becky 
was angry with Tom and quickly went out of the room. 
Then the children and the teacher came into the room 
and went to their places. The

### VectorStoreRetriever

리트리버는 벡터DB의 검색 기능을 표준화하고 추상화하여 LangChain 생태계에서 재사용성을 높이는 어댑터(Adapter) 역할을 수행한다.

벡터 저장소를 **`Retriever`라는 표준 인터페이스(Runnable)로 변환**한 뒤 실행하는 방식이다.

단순 유사도 검색뿐만 아니라, `search_type` 설정을 통해 **MMR(다양성 확보), 임계값 필터링(score_threshold)** 등 고급 검색 로직을 쉽게 적용할 수 있다.

**LCEL(LangChain Expression Language)** 파이프라인(`chain = retriever | llm`)에 즉시 통합 가능하다. 코드 수정 없이 검색 알고리즘만 교체하기 쉽다.

In [ ]:
# VectorStore를 Retriever 인터페이스 변환
retriever = vector_db.as_retriever(
    search_type = 'similarity',  # 검색 방식 : 코사인 유사도
    search_kwargs = {  # 검색 파라미터 묶음
        'k': 3  # 상위 3개
    }
)

# 질의 실행 -> list[Document]
search_results = retriever.invoke('마을 무덤의 남자는 누가 죽였는가?')

for i, doc in enumerate(search_results, 1):
    # 사람이 보는 페이지 : 본문 내용
    print(f"{i}번째 {doc.metadata['page_label']} \npage : {doc.page_content}")

1번째 19 
page : A man asked him, “Where were you on the night of  
June 17th?” 
“I was in the graveyard,” Tom answered. 
“Did you see any people there?” the man asked: 
“Yes. Injun Joe, the doctor, and Muff Potter were there. 
They didn’t see me because I was behind some big trees.” 
“What did you see?” the man asked. 
“Injun Joe and the doctor talked angrily,” Tom 
answered. “Then Injun Joe killed the doctor with his knife. 
Muff Potter didn’t do it.” 
The people at the trial were surprised. Injun Joe quickly 
went out of the building. 
Tom and Huck were very afraid. Tom said, “Now Injun 
Joe knows about us. He can kill us, too.” 
Many people wanted to hear about the boys’ adventure 
in the graveyard. Tom liked talking about it. He was 
happy, too, because he helped Muff Potter. But he didn’t 
sleep well because he was afraid of Injun Joe. 
 
Chapter 7    Injun Joe’s Treasure 
 
One Saturday afternoon, Tom wanted to have an adventure 
because he didn’t want to think about Injun Joe. He

In [37]:
# 벡터 검색(Retriever) 결과를 Context에 넣고, PDF 기반 RAG 답변을 생성하는 코드
retriever = vector_db.as_retriever(
    search_type = 'similarity',  # 검색 방식 : 코사인 유사도
    search_kwargs = {  # 검색 파라미터 묶음
        'k': 3  # 상위 3개
    }
)

prompt = PromptTemplate.from_template('''
    사용자의 질문에 Context 기반으로 답변하세요. 모르는 내용은 모른다고 답변하세요.
    Context : {context}
    Question : {question}
''')  # 프롬프트 지정

llm = init_chat_model('openai:gpt-5.6-luna')  # 사용할 LLM 모델 지정
output_parser = StrOutputParser()  # 최종 출력은 텍스트 형식


# question은 입력값 그대로 전달, context는 검색 + 문서 합치기 | 최종 프롬프트 완성 | LLM 호출 | 문자열 파싱
chain = (
    {'question' : RunnablePassthrough(), 'context' : retriever | format_docs } | prompt | llm |output_parser
)

print(chain.invoke('선생님의 해부학 책은 누가 찢었는가?'))

선생님의 해부학 책은 톰 소여가 찢었습니다. 톰이 “내가 그랬어요. 선생님의 책을 찢었어요.”라고 말했습니다.


### 음식 리뷰 조회 RAG
- 데이터셋 : fine_food_reviews_1k.csv
- 벡터DB 구성
- Retriever + llm 체인으로 리뷰 조회하는 기능

In [38]:
import pandas as pd

df = pd.read_csv('fine_food_reviews_1k.csv')
data = df['Text'].to_list()

data

['Wanted to save some to bring to my Chicago family but my North Carolina family ate all 4 boxes before I could pack. These are excellent...could serve to anyone',
 'Not pleased at all. When I opened the box, most of the rings were broken in pieces. A total waste of money.',
 'I\'m not sure that custard is really custard without eggs.  But this comes close.  I got it for use in a "Vegan pancake" recipe.  We were having houseguests who were Vegan and I wanted to make some special breakfasts while they were here.  One of the cooking/recipe sites had a recipe using this and there were lots of great reviews.  I tried the recipe and it turned out like wallpaper paste -- yuck!<br />However, the  so-called custard isn\'t so bad.  I think it\'s probably just cornstarch and annatto (yellow coloring with a slight flavor).  It\'s fun playing with it.  You could dress it up with fruit.  Seems to come out on the thin side when you make it as directed, so I use less milk because I like my custards t

In [43]:
vector_store = FAISS.from_texts(data, embeddings)  # 리뷰 텍스트들을 임베딩 -> FAISS 벡터스토어 생성

# Retriever Chain 구성 (코사인 유사도, 상위 10개 리뷰 검색)
retriever = vector_store.as_retriever(
    search_type = 'similarity',  # 검색 방식 : 코사인 유사도
    search_kwargs = {  # 검색 파라미터 묶음
        'k': 10  # 상위 3개
    }
)

# prompt Chain 요소 구성 (context, question 입력받음)
prompt = PromptTemplate.from_template('''
    Context : {context}
    Question : {question}
''')

# llm / 문자열 반환 parser 생성
llm = init_chat_model('openai:gpt-5.6-luna')  # 사용할 LLM 모델 지정
output_parser = StrOutputParser()  # 최종 출력은 텍스트 형식

# 체인 구성 : question은 입력값 그대로 전달, context는 검색 + 문서 합치기 | 최종 프롬프트 완성 | LLM 호출 | 문자열 파싱
chain = (
    {'question': RunnablePassthrough(), 'context': retriever | format_docs } | prompt | llm | output_parser
)

# fresh fruit 관련 답변 출력
print(chain.invoke('신선한 과일이 있나요?'))

print(chain.invoke('팝콘이 맛있나요?'))

제공된 내용만으로는 신선한 과일이 있는지 확인할 수 없습니다.
제공된 리뷰에는 팝콘에 대한 내용이 없어 맛있는지 판단하기 어렵습니다.
